In [3]:
import pandas as pd
import numpy as np
import csv
import json
import sys
from lxml import objectify

## Working with Other Delimited Formats — Python `csv` Module

When `pandas.read_csv` cannot handle a malformed or unusual file, Python's built-in `csv` module provides lower-level control.

- Works for any **single-character delimiter**.
- `csv.reader(file)` yields each row as a list of strings with quote characters stripped.

### Manual Parsing Pattern

```python
with open(file) as f:
    lines = list(csv.reader(f))

header, values = lines[0], lines[1:]
data_dict = {h: v for h, v in zip(header, zip(*values))}
```

- `zip(*values)` transposes rows into columns.
- The resulting dict maps each column name to a tuple of its values.

### Custom Dialects

Define a subclass of `csv.Dialect` to describe a non-standard format, then pass it to `csv.reader` or `csv.writer`:

```python
class my_dialect(csv.Dialect):
    lineterminator = "\n"
    delimiter = ";"
    quotechar = '"'
    quoting = csv.QUOTE_MINIMAL
```

Individual options can also be passed directly as keyword arguments without subclassing:

```python
reader = csv.reader(f, delimiter="|")
```

### `csv.Dialect` Options

| Option | Description |
|--------|-------------|
| `delimiter` | Field separator; default `","` |
| `lineterminator` | Line ending for writing; default `"\r\n"` |
| `quotechar` | Quote character for fields with special characters; default `'"'` |
| `quoting` | Quoting convention: `QUOTE_ALL`, `QUOTE_MINIMAL`, `QUOTE_NONNUMERIC`, `QUOTE_NONE` |
| `skipinitialspace` | Ignore whitespace after each delimiter; default `False` |
| `doublequote` | Whether to double the quote character inside a field |
| `escapechar` | Escape string for the delimiter when `quoting=QUOTE_NONE` |

### Writing with `csv.writer`

`csv.writer` takes an open writable file and accepts the same dialect options as `csv.reader`. Use `writerow` to write one row at a time.

## Key Points

- Use the `csv` module when `read_csv` cannot handle a malformed file.
- `csv.reader` yields rows as string lists — quoting is handled automatically.
- `zip(*values)` transposes a list of rows into a dict of columns.
- Custom formats can be described with a `csv.Dialect` subclass or inline keyword arguments.
- For multi-character or complex delimiters, use `str.split` or `re.split` instead.

In [4]:
# Iterate with csv.reader — quotes are stripped automatically
with open("../examples/ex7.csv") as f:
    reader = csv.reader(f)
    for line in reader:
        print(line)

['a', 'b', 'c']
['1', '2', '3']
['1', '2', '3']


In [5]:
# Manually build a column-oriented dict from the CSV
with open("../examples/ex7.csv") as f:
    lines = list(csv.reader(f))

header, values = lines[0], lines[1:]

data_dict = {h: v for h, v in zip(header, zip(*values))}

data_dict

{'a': ('1', '1'), 'b': ('2', '2'), 'c': ('3', '3')}

In [6]:
# Writing with csv.writer
with open("../examples/mydata.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(("one", "two", "three"))
    writer.writerow(("1", "2", "3"))
    writer.writerow(("4", "5", "6"))
    writer.writerow(("7", "8", "9"))

pd.read_csv("../examples/mydata.csv")

,one,two,three
0,1,2,3
1,4,5,6
2,7,8,9


## JSON Data

JSON (JavaScript Object Notation) is a standard format for data exchange.

### JSON vs Python Types

| JSON | Python |
|------|--------|
| object `{}` | dict |
| array `[]` | list |
| string | str |
| number | int / float |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

JSON is nearly valid Python but differs in two key ways:
- `null` instead of `None`.
- No trailing commas allowed in arrays or objects.

### Python `json` Module

- `json.loads(string)` → parse a JSON string into Python objects.
- `json.dumps(obj)` → convert a Python object back to a JSON string.

### JSON → DataFrame

- Pass a list of dicts (parsed from JSON) directly to the `DataFrame` constructor.
- Use `pd.read_json(file)` for JSON files where each array element is a row.

### DataFrame → JSON

- `df.to_json()` — default: column-oriented `{col: {index: value}}`.
- `df.to_json(orient="records")` — row-oriented `[{col: value, ...}, ...]`.

## Key Points

- `json.loads` / `json.dumps` convert between JSON strings and Python objects.
- A list of dicts from JSON can be passed directly to the `DataFrame` constructor.
- `pd.read_json` reads JSON files where each element maps to a row.
- `to_json(orient="records")` produces a row-per-object format suitable for APIs.

In [7]:
obj = """
{"name": "Wes",
 "cities_lived": ["Akron", "Nashville", "New York", "San Francisco"],
 "pet": null,
 "siblings": [{"name": "Scott", "age": 34, "hobbies": ["guitars", "soccer"]},
              {"name": "Katie", "age": 42, "hobbies": ["diving", "art"]}]
}
"""

result = json.loads(obj)

result

{'name': 'Wes',
 'cities_lived': ['Akron', 'Nashville', 'New York', 'San Francisco'],
 'pet': None,
 'siblings': [{'name': 'Scott', 'age': 34, 'hobbies': ['guitars', 'soccer']},
  {'name': 'Katie', 'age': 42, 'hobbies': ['diving', 'art']}]}

In [8]:
asjson = json.dumps(result)

asjson

'{"name": "Wes", "cities_lived": ["Akron", "Nashville", "New York", "San Francisco"], "pet": null, "siblings": [{"name": "Scott", "age": 34, "hobbies": ["guitars", "soccer"]}, {"name": "Katie", "age": 42, "hobbies": ["diving", "art"]}]}'

In [9]:
# Build a DataFrame from a nested JSON list
siblings = pd.DataFrame(result["siblings"], columns=["name", "age"])

siblings

,name,age
0,Scott,34
1,Katie,42


In [10]:
# pd.read_json: each array element becomes a row
data = pd.read_json("../examples/example.json")

data

,a,b,c
0,1,2,3
1,4,5,6
2,7,8,9


In [11]:
# Default to_json: column-oriented
data.to_json(sys.stdout)

{"a":{"0":1,"1":4,"2":7},"b":{"0":2,"1":5,"2":8},"c":{"0":3,"1":6,"2":9}}

In [12]:
# orient="records": row-oriented list of objects
data.to_json(sys.stdout, orient="records")

[{"a":1,"b":2,"c":3},{"a":4,"b":5,"c":6},{"a":7,"b":8,"c":9}]

## HTML: Web Scraping with `pd.read_html`

`pandas.read_html` automatically finds and parses all `<table>` tags in an HTML document.

- Returns a **list of DataFrames** — one per table found.
- Uses `lxml`, `BeautifulSoup4`, or `html5lib` under the hood.
- Install dependencies: `conda install lxml beautifulsoup4 html5lib`.

Date columns loaded as strings can be converted with `pd.to_datetime`, after which `.dt` accessor attributes (like `.dt.year`) become available.

## Key Points

- `pd.read_html(file_or_url)` returns a list of DataFrames, one per `<table>`.
- Select the table you want by index: `tables[0]`.
- Date columns come in as strings — use `pd.to_datetime` to convert them.

In [13]:
tables = pd.read_html("../examples/fdic_failed_bank_list.html")

print(f"{len(tables)} table(s) found")

failures = tables[0]

failures.head()

1 table(s) found


,Bank Name,City,ST,CERT,Acquiring Institution,Closing Date,Updated Date
0,Allied Bank,Mulberry,AR,91,Today's Bank,"September 23, 2016","November 17, 2016"
1,The Woodbury Banking Company,Woodbury,GA,11297,United Bank,"August 19, 2016","November 17, 2016"
2,First CornerStone Bank,King of Prussia,PA,35312,First-Citizens Bank & Trust Company,"May 6, 2016","September 6, 2016"
3,Trust Company Bank,Memphis,TN,9956,The Bank of Fayette County,"April 29, 2016","September 6, 2016"
4,North Milwaukee State Bank,Milwaukee,WI,20364,First-Citizens Bank & Trust Company,"March 11, 2016","June 16, 2016"


In [14]:
# Convert date strings and count failures by year
close_timestamps = pd.to_datetime(failures["Closing Date"])

close_timestamps.dt.year.value_counts()

Closing Date
2010    157
2009    140
2011     92
2012     51
2008     25
2013     24
2014     18
2002     11
2015      8
2016      5
2004      4
2001      4
2007      3
2003      3
2000      2
Name: count, dtype: int64

## XML Parsing

XML is a general hierarchical data format — similar in structure to HTML but more flexible.

### Approach 1: `lxml.objectify` (Manual)

Use when you need fine-grained control over which elements and fields to extract:

1. Parse the file with `objectify.parse(f)` and get the root with `.getroot()`.
2. Iterate over child elements using `root.ELEMENT_NAME`.
3. For each element, call `.getchildren()` to access its child tags.
4. Use `.tag` for the tag name and `.pyval` for the value (auto-converted to Python type).
5. Build a list of dicts, then pass to `pd.DataFrame`.

### Approach 2: `pd.read_xml` (Simple)

For straightforward XML, `pd.read_xml` reads the file in a single call — equivalent to the manual `lxml` approach but without any boilerplate.

- Includes all fields; skip unwanted columns after reading.
- For complex documents, see `pd.read_xml` docstring for selection and filter options.

## Key Points

- `lxml.objectify` gives full control — useful when skipping fields or handling nested structures.
- `.pyval` on an `lxml` element automatically converts the text to the appropriate Python type.
- `pd.read_xml` is a one-liner equivalent for simple XML tables.
- Both approaches produce a DataFrame with the same data.

In [15]:
path = "../datasets/mta_perf/Performance_MNR.xml"

with open(path) as f:
    parsed = objectify.parse(f)

root = parsed.getroot()

In [16]:
# Build a list of dicts, skipping unwanted fields
skip_fields = ["PARENT_SEQ", "INDICATOR_SEQ", "DESIRED_CHANGE", "DECIMAL_PLACES"]

data = []
for elt in root.INDICATOR:
    el_data = {}
    for child in elt.getchildren():
        if child.tag in skip_fields:
            continue
        el_data[child.tag] = child.pyval
    data.append(el_data)

perf = pd.DataFrame(data)

perf.head()

,AGENCY_NAME,INDICATOR_NAME,DESCRIPTION,PERIOD_YEAR,PERIOD_MONTH,CATEGORY,FREQUENCY,INDICATOR_UNIT,YTD_TARGET,YTD_ACTUAL,MONTHLY_TARGET,MONTHLY_ACTUAL
0,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,1,Service Indicators,M,%,95.0,96.9,95.0,96.9
1,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,2,Service Indicators,M,%,95.0,96.0,95.0,95.0
2,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,3,Service Indicators,M,%,95.0,96.3,95.0,96.9
3,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,4,Service Indicators,M,%,95.0,96.8,95.0,98.3
4,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,5,Service Indicators,M,%,95.0,96.6,95.0,95.8


In [17]:
# pd.read_xml: equivalent one-liner (includes all fields)
perf2 = pd.read_xml(path)

perf2.head()

,INDICATOR_SEQ,PARENT_SEQ,AGENCY_NAME,INDICATOR_NAME,DESCRIPTION,PERIOD_YEAR,PERIOD_MONTH,CATEGORY,FREQUENCY,DESIRED_CHANGE,INDICATOR_UNIT,DECIMAL_PLACES,YTD_TARGET,YTD_ACTUAL,MONTHLY_TARGET,MONTHLY_ACTUAL
0,28445,NaN,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,1,Service Indicators,M,U,%,1,95.00,96.90,95.00,96.90
1,28445,NaN,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,2,Service Indicators,M,U,%,1,95.00,96.00,95.00,95.00
2,28445,NaN,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,3,Service Indicators,M,U,%,1,95.00,96.30,95.00,96.90
3,28445,NaN,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,4,Service Indicators,M,U,%,1,95.00,96.80,95.00,98.30
4,28445,NaN,Metro-North Railroad,On-Time Performance (West of Hudson),Percent of commuter trains that arrive at thei...,2008,5,Service Indicators,M,U,%,1,95.00,96.60,95.00,95.80
